# Complete Transit Nodes Processing Pipeline

This notebook integrates the complete workflow:

## Part 1: H3 Hexagon Processing (Steps 1-4)
1. Load transit nodes from CSV
2. Assign H3 hexagonal indices
3. Group hexagons by 120m proximity (edge-to-edge)
4. Geocode addresses (optional)
5. Export grouped hexagons

## Part 2: Demand Data Processing (Steps 5-8)
5. Load grouped hexagons
6. Tag with spatial information (metro areas, districts)
7. Add daily demand from regional transport models
8. Create aggregated hub groups with demand statistics

## Part 3: Influence Area Processing (Steps 9-11) [Optional]
9. Create buffer zones (0-600m, 600-1000m, 1000-1200m)
10. Calculate population and employment from TAZ data
11. Identify proximity to bus terminals

## Part 4: Final Export and Analysis (Step 12)
12. Export complete dataset with analysis

**Total Runtime:** ~8-10 minutes for complete pipeline
- Part 1: ~2 minutes (without geocoding)
- Part 2: ~3 minutes
- Part 3: ~2-3 minutes
- Part 4: ~1 minute

---
# PART 1: H3 HEXAGON PROCESSING
---

## Step 1.1: Setup and Configuration

In [ ]:
# Install required packages (uncomment if needed)
# !pip install h3 geopandas pandas shapely geopy openpyxl

In [ ]:
import h3
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon
from shapely import wkt
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import warnings
import os
warnings.filterwarnings('ignore')

print("✓ All libraries loaded successfully!")

## Step 1.2: Configure File Paths for Part 1

In [ ]:
# ============================================================================
# PART 1 CONFIGURATION - H3 HEXAGON PROCESSING
# ============================================================================

# Input: Transit nodes CSV
INPUT_NODES_CSV = '/path/to/All_nodes+lines.csv'

# Input: Lines and Planned Mode CSV (for mode determination)
LINES_MODE_CSV = '/path/to/Lines_and_Planned_Mode.csv'

# Output: H3 hexagons with groups
OUTPUT_H3_HEXAGONS = '/path/to/output/transit_h3_hexagons.csv'

# Processing parameters
H3_RESOLUTION = 10        # H3 resolution (10 = ~15m hexagons)
BUFFER_DISTANCE = 120     # Buffer distance in meters for grouping (edge-to-edge)
SKIP_GEOCODING = True     # Set to False to geocode addresses (much slower)

# Coordinate reference systems
CRS_PROJECTED = "EPSG:2039"  # Israel TM Grid (for meter-based operations)
CRS_WGS84 = "EPSG:4326"      # WGS84 (for H3 and geocoding)

print("Part 1 Configuration:")
print(f"  H3 Resolution: {H3_RESOLUTION}")
print(f"  Buffer Distance: {BUFFER_DISTANCE}m (edge-to-edge)")
print(f"  Skip Geocoding: {SKIP_GEOCODING}")
print(f"  Input: {INPUT_NODES_CSV}")
print(f"  Output: {OUTPUT_H3_HEXAGONS}")

## Step 1.3: Load Transit Nodes Data

In [ ]:
print("Loading transit nodes data...")

# Read CSV with pandas
df = pd.read_csv(INPUT_NODES_CSV, encoding='windows-1255')

# Check if geometry column exists
if 'geometry' in df.columns:
    # If geometry exists as WKT string, convert it
    df['geometry'] = df['geometry'].apply(lambda x: wkt.loads(x) if pd.notna(x) else None)
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs=CRS_PROJECTED)
elif 'X' in df.columns and 'Y' in df.columns:
    # Create geometry from X, Y coordinates
    geometry = [Point(x, y) for x, y in zip(df['X'], df['Y'])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=CRS_PROJECTED)
else:
    raise ValueError("CSV must contain either 'geometry' column or both 'X' and 'Y' columns")

print(f"✓ Loaded {len(gdf)} records")
print(f"  CRS: {gdf.crs}")
print(f"  Columns: {list(gdf.columns)}")
print(f"\nFirst 3 rows:")
gdf.head(3)

## Step 1.4: Assign H3 Indices and Aggregate Lines

In [ ]:
print("Step 1.4: Loading Lines and Planned Mode data, then assigning H3 indices...")

# ============================================================================
# IMPORTANT: Configure path to Lines_and_Planned_Mode.csv
# ============================================================================
LINES_MODE_CSV = '/path/to/Lines_and_Planned_Mode.csv'  # UPDATE THIS PATH!

print(f"  Loading mode data from: {LINES_MODE_CSV}")
lines_mode = pd.read_csv(LINES_MODE_CSV, encoding='windows-1255')
print(f"  ✓ Loaded {len(lines_mode)} line records")
print(f"  Columns: {list(lines_mode.columns)}")

# Convert to WGS84 for H3
gdf_wgs84 = gdf.to_crs(CRS_WGS84)

# Extract lat/lon
gdf_wgs84['lat'] = gdf_wgs84.geometry.y
gdf_wgs84['lon'] = gdf_wgs84.geometry.x

# Assign H3 index
gdf_wgs84['h3_index'] = gdf_wgs84.apply(
    lambda row: h3.latlng_to_cell(row['lat'], row['lon'], H3_RESOLUTION),
    axis=1
)

print(f"  ✓ Assigned H3 indices to {len(gdf_wgs84)} records")
print(f"  Unique H3 hexagons (before filtering): {gdf_wgs84['h3_index'].nunique()}")

# ============================================================================
# Filter out old Metronit lines (Haifa area, starting with 'm')
# ============================================================================
print("\n  Filtering out old Metronit lines...")
remove_old_metronit = []
if 'Area' in lines_mode.columns:
    haifa_lines = lines_mode[lines_mode['Area'] == 'Haifa']['Line_ModelName'].tolist()
    for line in haifa_lines:
        if str(line)[:1] == 'm':
            remove_old_metronit.append(line)
            
    if remove_old_metronit:
        print(f"    Removing old Metronit: {', '.join(map(str, remove_old_metronit))}")
        gdf_wgs84 = gdf_wgs84[~gdf_wgs84['LINE_ID'].isin(remove_old_metronit)]
        print(f"    ✓ After filtering: {len(gdf_wgs84)} records")
    else:
        print(f"    No old Metronit lines found")
else:
    print(f"    ⚠ 'Area' column not found in Lines_and_Planned_Mode.csv, skipping Metronit filter")

# ============================================================================
# Filter out Netanya LRT151/152 lines
# ============================================================================
print("\n  Filtering out Netanya LRT151/152 lines...")
remove_netanya15 = []
if 'Area' in lines_mode.columns:
    netanya_lines = lines_mode[lines_mode['Area'] == 'Netanya']['Line_ModelName'].tolist()
    for line in netanya_lines:
        if line == 'LRT151' or line == 'LRT152':
            remove_netanya15.append(line)
            
    if remove_netanya15:
        print(f"    Removing Netanya lines: {', '.join(map(str, remove_netanya15))}")
        gdf_wgs84 = gdf_wgs84[~gdf_wgs84['LINE_ID'].isin(remove_netanya15)]
        print(f"    ✓ After filtering: {len(gdf_wgs84)} records")
    else:
        print(f"    No Netanya LRT151/152 lines found")
else:
    print(f"    ⚠ 'Area' column not found, skipping Netanya filter")

# ============================================================================
# Merge with Lines and Planned Mode to get Mode_Planned
# ============================================================================
print("\n  Merging with mode data...")
gdf_with_mode = gdf_wgs84.merge(
    lines_mode, 
    left_on='LINE_ID', 
    right_on='Line_ModelName', 
    how='left',
    validate='many_to_many'
)

print(f"  ✓ Merged data: {len(gdf_with_mode)} records")

# Check if Mode_Planned column exists after merge
if 'Mode_Planned' not in gdf_with_mode.columns:
    print("  ⚠ WARNING: 'Mode_Planned' column not found after merge!")
    print(f"  Available columns: {list(gdf_with_mode.columns)}")
    print("  Using 'Bus' as default mode for all lines")
    gdf_with_mode['Mode_Planned'] = 'Bus'

# ============================================================================
# Group by h3_index, node, and Mode_Planned, then aggregate lines
# ============================================================================
print("\n  Aggregating by H3 hexagon and mode...")

h3_grouped = gdf_with_mode.groupby(['h3_index', 'node', 'Mode_Planned']).agg({
    'Line_ModelName': ['nunique', lambda x: list(x.unique())]
}).reset_index()

# Flatten column names
h3_grouped.columns = ['h3_index', 'node', 'Mode_Planned', 'Line_Nunique', 'Line_Unique']

print(f"  ✓ Created {len(h3_grouped)} unique H3-node-mode combinations")

# ============================================================================
# Further aggregate by h3_index only (combining all modes)
# ============================================================================
print("\n  Final aggregation by H3 hexagon...")

h3_final = h3_grouped.groupby('h3_index').agg({
    'node': 'first',
    'Mode_Planned': lambda x: list(x.unique()),  # List of all modes
    'Line_Nunique': 'sum',  # Total unique lines across all modes
    'Line_Unique': lambda x: list(set([item for sublist in x for item in sublist]))  # Flatten and deduplicate lines
}).reset_index()

print(f"  ✓ Aggregated to {len(h3_final)} unique hexagons")

# ============================================================================
# Create geometry from H3 indices
# ============================================================================
def h3_to_polygon(h3_index):
    """Convert H3 index to Shapely Polygon."""
    boundary = h3.cell_to_boundary(h3_index)
    # H3 returns (lat, lon), need (lon, lat) for Shapely
    return Polygon([(lon, lat) for lat, lon in boundary])

print("  Creating hexagon geometries...")
h3_final['geometry'] = h3_final['h3_index'].apply(h3_to_polygon)

# Convert to GeoDataFrame
gdf_h3 = gpd.GeoDataFrame(h3_final, geometry='geometry', crs=CRS_WGS84)

print(f"\n✓ Step 1.4 complete! Created {len(gdf_h3)} hexagons")
print(f"  Modes distribution:")
if 'Mode_Planned' in gdf_h3.columns:
    all_modes = [mode for modes_list in gdf_h3['Mode_Planned'] for mode in modes_list]
    mode_counts = pd.Series(all_modes).value_counts()
    for mode, count in mode_counts.items():
        print(f"    {mode}: {count}")

gdf_h3.head(3)

## Step 1.5: Create Groups Based on 120m Buffer (Edge-to-Edge)

In [ ]:
print(f"Step 1.5: Creating groups based on {BUFFER_DISTANCE}m buffer...")
print("Note: Measuring edge-to-edge distance (border to border)")
print("Note: Using transitive grouping (A near B, B near C → all same group)")

# Convert to projected CRS for meter-based operations
gdf_proj = gdf_h3.to_crs(CRS_PROJECTED).copy()
gdf_proj = gdf_proj.reset_index(drop=True)

n = len(gdf_proj)
print(f"  Processing {n} hexagons...")

In [ ]:
# Build spatial index for efficient querying
print("  Building spatial index...")
sindex = gdf_proj.sindex

# Union-Find data structure
parent = list(range(n))

def find(x):
    """Find root with path compression."""
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    """Union two components."""
    root_x = find(x)
    root_y = find(y)
    if root_x != root_y:
        parent[root_y] = root_x

print("  Union-Find structure initialized")

In [ ]:
# Find all pairs of hexagons within buffer_distance (edge to edge)
print("  Finding neighbors within buffer distance (edge-to-edge)...")
pairs_found = 0

# Add small tolerance for touching hexagons (floating point precision)
tolerance = 0.1  # 10cm tolerance for "touching"
effective_distance = BUFFER_DISTANCE + tolerance

for idx1 in range(n):
    geom1 = gdf_proj.iloc[idx1].geometry
    
    # Query spatial index for potential neighbors
    minx, miny, maxx, maxy = geom1.bounds
    search_bounds = (
        minx - BUFFER_DISTANCE,
        miny - BUFFER_DISTANCE,
        maxx + BUFFER_DISTANCE,
        maxy + BUFFER_DISTANCE
    )
    
    possible_neighbors = list(sindex.intersection(search_bounds))
    
    # Check actual edge-to-edge distance
    for idx2 in possible_neighbors:
        if idx2 <= idx1:  # Avoid checking pairs twice
            continue
        
        geom2 = gdf_proj.iloc[idx2].geometry
        distance = geom1.distance(geom2)
        
        if distance <= effective_distance:
            union(idx1, idx2)
            pairs_found += 1
    
    if (idx1 + 1) % 100 == 0:
        print(f"    Processed {idx1 + 1}/{n} hexagons...")

print(f"  ✓ Found {pairs_found} neighbor pairs within {BUFFER_DISTANCE}m (edge-to-edge)")

In [ ]:
# Assign group labels
print("  Assigning group IDs...")
root_to_group = {}
group_counter = 0

for idx in range(n):
    root = find(idx)
    if root not in root_to_group:
        root_to_group[root] = group_counter
        group_counter += 1

labels = [root_to_group[find(idx)] for idx in range(n)]
gdf_h3['group'] = labels
n_groups = len(root_to_group)

# Statistics
group_sizes = gdf_h3.groupby('group').size()

print(f"\n✓ Step 1.5 complete!")
print(f"  Created {n_groups} groups from {len(gdf_h3)} hexagons")
print(f"  Single hexagon groups: {(group_sizes == 1).sum()}")
print(f"  Multi-hexagon groups: {(group_sizes > 1).sum()}")
print(f"  Largest group: {group_sizes.max()} hexagons")
print(f"  Average group size: {group_sizes.mean():.2f} hexagons")

## Step 1.6: Geocode Addresses (Optional)

In [ ]:
if not SKIP_GEOCODING:
    print("Step 1.6: Geocoding addresses...")
    print("Note: This step uses Nominatim and may take several minutes.")
    print("      Rate limited to 1 request per second.\n")
    
    # Initialize geocoder
    geolocator = Nominatim(user_agent='transit_h3_processor')
    geocode = RateLimiter(geolocator.reverse, min_delay_seconds=1)
    
    # Get centroids in WGS84
    gdf_wgs84_final = gdf_h3.to_crs(CRS_WGS84)
    centroids = gdf_wgs84_final.geometry.centroid
    
    addresses = []
    total = len(centroids)
    
    for idx, centroid in enumerate(centroids):
        try:
            location = geocode(f"{centroid.y}, {centroid.x}", language='he')
            address = location.address if location else "Address not found"
            addresses.append(address)
            
            if (idx + 1) % 10 == 0:
                print(f"  Geocoded {idx + 1}/{total} addresses ({(idx+1)/total*100:.1f}%)")
        except Exception as e:
            print(f"  Error geocoding index {idx}: {e}")
            addresses.append("Geocoding error")
    
    gdf_h3['address'] = addresses
    print(f"\n✓ Step 1.6 complete! Geocoded {len(addresses)} locations")
    
else:
    print("Step 1.6: Skipping geocoding (as configured)")
    gdf_h3['address'] = 'Not geocoded'
    print("✓ Step 1.6 complete (skipped)")

## Step 1.7: Export H3 Hexagons with Groups

In [ ]:
print(f"Step 1.7: Exporting H3 hexagons to {OUTPUT_H3_HEXAGONS}...")

# Select final columns
output_columns = [
    'h3_index', 'node', 'Mode_Planned', 'Line_Nunique', 
    'Line_Unique', 'geometry', 'group', 'address'
]

gdf_output_part1 = gdf_h3[output_columns].copy()

# CRITICAL: Clean node column to ensure it's a single integer value
# This prevents '[np.int64(123)]' format in CSV that breaks Part 2
print("  Cleaning node column...")
if 'node' in gdf_output_part1.columns:
    # If node is somehow a list, take the first value
    gdf_output_part1['node'] = gdf_output_part1['node'].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x
    )
    # Convert numpy types to Python int
    gdf_output_part1['node'] = gdf_output_part1['node'].apply(
        lambda x: int(x) if pd.notna(x) else x
    )
    print(f"  ✓ Node column cleaned: {gdf_output_part1['node'].dtype}")

# Ensure output directory exists
os.makedirs(os.path.dirname(OUTPUT_H3_HEXAGONS) if os.path.dirname(OUTPUT_H3_HEXAGONS) else '.', exist_ok=True)

# Export
if OUTPUT_H3_HEXAGONS.endswith('.csv'):
    gdf_output_part1['geometry'] = gdf_output_part1['geometry'].apply(lambda x: x.wkt)
    gdf_output_part1.to_csv(OUTPUT_H3_HEXAGONS, index=False, encoding='utf-8-sig')
elif OUTPUT_H3_HEXAGONS.endswith('.geojson'):
    gdf_output_part1.to_file(OUTPUT_H3_HEXAGONS, driver='GeoJSON')
elif OUTPUT_H3_HEXAGONS.endswith('.gpkg'):
    gdf_output_part1.to_file(OUTPUT_H3_HEXAGONS, driver='GPKG')
else:
    gdf_output_part1.to_file(OUTPUT_H3_HEXAGONS + '.geojson', driver='GeoJSON')

print(f"\n✓ Step 1.7 complete! File saved successfully.")
print(f"\n" + "="*80)
print("PART 1 COMPLETE - H3 HEXAGON PROCESSING")
print("="*80)
print(f"\nOutput: {OUTPUT_H3_HEXAGONS}")
print(f"Records: {len(gdf_output_part1)} INDIVIDUAL hexagons")
print(f"Groups: {gdf_output_part1['group'].nunique()} proximity groups identified")
print(f"\nColumns: {list(gdf_output_part1.columns)}")
print(f"\nNote: Grouping will happen in Part 2 AFTER demand assignment")
print(f"Ready for Part 2: Demand Data Processing")

---
# PART 2: DEMAND DATA PROCESSING
---

This part requires additional files from the hub demand processor.
Please ensure you have the following Python module:
- `hub_demand_processor.py`

And the following data files:
- Demand Excel file (with regional model sheets)
- Metro shapefile
- Districts shapefile

## Step 2.1: Configure Paths for Part 2

In [ ]:
# ============================================================================
# PART 2 CONFIGURATION - DEMAND DATA PROCESSING
# ============================================================================

# NOTE: This section requires hub_demand_processor.py module
# If you don't have it, you can skip Part 2 and use the H3 hexagons from Part 1

# Input: H3 hexagons from Part 1 (automatically set)
INPUT_H3_FOR_DEMAND = OUTPUT_H3_HEXAGONS

# Input: Demand data files
DEMAND_EXCEL = '/path/to/demand_data.xlsx'
METRO_SHP = '/path/to/metro.shp'
DISTRICTS_SHP = '/path/to/districts.shp'

# Output: Hubs with demand
OUTPUT_HUBS_WITH_DEMAND = '/path/to/output/groups_hubs_with_demand.csv'
OUTPUT_GROUPED_HUBS = '/path/to/output/Grouped_Hubs_Final.csv'

print("Part 2 Configuration:")
print(f"  Input H3: {INPUT_H3_FOR_DEMAND}")
print(f"  Demand Excel: {DEMAND_EXCEL}")
print(f"  Output (ungrouped): {OUTPUT_HUBS_WITH_DEMAND}")
print(f"  Output (grouped): {OUTPUT_GROUPED_HUBS}")
print(f"\nTo run Part 2, you need hub_demand_processor.py")
print(f"Skip to Part 3 if you only want H3 hexagons.")

## Step 2.2: Load Hub Demand Processor Module

In [ ]:
# Import hub demand processor
# NOTE: This requires hub_demand_processor.py to be in the same directory
# or in your Python path

try:
    from hub_demand_processor import DemandDataProcessor
    print("✓ Hub demand processor module loaded successfully!")
    PART2_AVAILABLE = True
except ImportError as e:
    print("✗ Could not load hub_demand_processor module")
    print(f"  Error: {e}")
    print(f"\n  To run Part 2, ensure hub_demand_processor.py is available.")
    print(f"  You can continue with Part 1 results only.")
    PART2_AVAILABLE = False

## Step 2.3: Initialize Demand Processor and Run Pipeline

In [ ]:
if PART2_AVAILABLE:
    print("Step 2.3: Running demand processing pipeline...")
    
    # Initialize processor
    processor = DemandDataProcessor()
    
    # Run full pipeline
    gdf_with_demand, grouped_hubs = processor.process_full_pipeline(
        hubs_csv=INPUT_H3_FOR_DEMAND,
        demand_excel=DEMAND_EXCEL,
        metro_shp=METRO_SHP,
        districts_shp=DISTRICTS_SHP,
        output_csv=OUTPUT_HUBS_WITH_DEMAND,
        grouped_output_csv=OUTPUT_GROUPED_HUBS
    )
    
    print(f"\n✓ Part 2 complete!")
    print(f"  Output files created:")
    print(f"    - {OUTPUT_HUBS_WITH_DEMAND}")
    print(f"    - {OUTPUT_GROUPED_HUBS}")
else:
    print("Step 2.3: Skipped (hub_demand_processor not available)")
    print("\nPart 1 output is available at:")
    print(f"  {OUTPUT_H3_HEXAGONS}")

---
# PART 3: INFLUENCE AREA PROCESSING (OPTIONAL)
---

This part adds demographic data (population, employment) based on buffer zones around each hub.

**What it does:**
- Creates 3 concentric buffer zones (0-600m, 600-1000m, 1000-1200m)
- Calculates population and employment from TAZ (Traffic Analysis Zones)
- Uses proportional allocation based on area overlap
- Identifies hubs near bus terminals

**Requirements:**
- `influence_area_processor.py` module
- TAZ shapefile with `POP_2050` and `EMPL_2050` columns
- (Optional) Bus terminals shapefile

**Expected runtime:** ~2-3 minutes for 1000-2000 hubs

## Step 3.1: Configure Paths for Part 3

In [ ]:
# ============================================================================
# PART 3 CONFIGURATION - INFLUENCE AREA PROCESSING
# ============================================================================

# Input: Grouped hubs from Part 2 (or Part 1 if Part 2 was skipped)
if PART2_AVAILABLE:
    INPUT_FOR_INFLUENCE = OUTPUT_GROUPED_HUBS
else:
    INPUT_FOR_INFLUENCE = OUTPUT_H3_HEXAGONS

# Input: TAZ shapefile with population and employment
TAZ_SHAPEFILE = '/path/to/TAZ_2050.shp'

# Input: Bus terminals shapefile (optional)
BUS_TERMINALS_SHP = '/path/to/bus_terminals.shp'  # Set to None if not available

# Output: Final hubs with all data
OUTPUT_FINAL = '/path/to/output/hubs_with_complete_data.csv'
OUTPUT_FINAL_EXCEL = '/path/to/output/hubs_with_complete_data.xlsx'

# Buffer zone distances (in meters)
BUFFER_ZONES = {
    'zone1': (0, 600),      # 0-600m
    'zone2': (600, 1000),   # 600-1000m
    'zone3': (1000, 1200)   # 1000-1200m
}

print("Part 3 Configuration:")
print(f"  Input hubs: {INPUT_FOR_INFLUENCE}")
print(f"  TAZ Shapefile: {TAZ_SHAPEFILE}")
print(f"  Bus Terminals: {BUS_TERMINALS_SHP if BUS_TERMINALS_SHP else 'Not provided (optional)'}")
print(f"  Output CSV: {OUTPUT_FINAL}")
print(f"  Output Excel: {OUTPUT_FINAL_EXCEL}")
print(f"  Buffer zones: {BUFFER_ZONES}")

## Step 3.2: Load Influence Area Processor Module

In [ ]:
# Import influence area processor
# NOTE: This requires influence_area_processor.py to be in the same directory
# or in your Python path

try:
    from influence_area_processor import InfluenceAreaProcessor
    print("✓ Influence area processor module loaded successfully!")
    PART3_AVAILABLE = True
except ImportError as e:
    print("✗ Could not load influence_area_processor module")
    print(f"  Error: {e}")
    print(f"\n  To run Part 3, ensure influence_area_processor.py is available.")
    print(f"  You can continue with results from Parts 1 and 2.")
    PART3_AVAILABLE = False

## Step 3.3: Check TAZ Data Availability

In [ ]:
import os

if PART3_AVAILABLE:
    print("Checking data availability...")
    
    # Check TAZ shapefile
    if os.path.exists(TAZ_SHAPEFILE):
        print(f"  ✓ TAZ shapefile found: {TAZ_SHAPEFILE}")
        TAZ_AVAILABLE = True
    else:
        print(f"  ✗ TAZ shapefile not found: {TAZ_SHAPEFILE}")
        print(f"    Update the path in Step 3.1 or skip Part 3")
        TAZ_AVAILABLE = False
    
    # Check bus terminals (optional)
    if BUS_TERMINALS_SHP and os.path.exists(BUS_TERMINALS_SHP):
        print(f"  ✓ Bus terminals found: {BUS_TERMINALS_SHP}")
        TERMINALS_AVAILABLE = True
    elif BUS_TERMINALS_SHP:
        print(f"  ⚠ Bus terminals not found: {BUS_TERMINALS_SHP}")
        print(f"    Will proceed without terminal proximity data")
        TERMINALS_AVAILABLE = False
    else:
        print(f"  ⚠ Bus terminals not configured (optional)")
        TERMINALS_AVAILABLE = False
    
    # Check input hubs
    if os.path.exists(INPUT_FOR_INFLUENCE):
        print(f"  ✓ Input hubs found: {INPUT_FOR_INFLUENCE}")
    else:
        print(f"  ✗ Input hubs not found: {INPUT_FOR_INFLUENCE}")
        print(f"    Make sure Parts 1 and 2 completed successfully")
        TAZ_AVAILABLE = False
else:
    print("Skipping data check (influence_area_processor not available)")
    TAZ_AVAILABLE = False

## Step 3.4: Initialize Processor and Run Pipeline

In [ ]:
if PART3_AVAILABLE and TAZ_AVAILABLE:
    print("Step 3.4: Running influence area processing pipeline...")
    print("="*80)
    
    # Initialize processor
    processor = InfluenceAreaProcessor()
    
    # Set custom buffer zones if specified
    processor.buffer_zones = BUFFER_ZONES
    
    # Run full pipeline
    try:
        final_gdf = processor.process_full_pipeline(
            hubs_csv=INPUT_FOR_INFLUENCE,
            taz_shp=TAZ_SHAPEFILE,
            terminals_shp=BUS_TERMINALS_SHP if TERMINALS_AVAILABLE else None,
            output_csv=OUTPUT_FINAL
        )
        
        print(f"\n✓ Part 3 complete!")
        print(f"  Final dataset: {len(final_gdf)} hub groups")
        print(f"  Output file: {OUTPUT_FINAL}")
        
        PART3_SUCCESS = True
        
    except Exception as e:
        print(f"\n✗ Error during influence area processing:")
        print(f"  {e}")
        print(f"\n  You can still use the output from Parts 1 and 2.")
        PART3_SUCCESS = False
        final_gdf = None
else:
    if not PART3_AVAILABLE:
        print("Step 3.4: Skipped (influence_area_processor not available)")
    elif not TAZ_AVAILABLE:
        print("Step 3.4: Skipped (TAZ data not available)")
    
    print("\nParts 1 and 2 output is available:")
    if PART2_AVAILABLE:
        print(f"  {OUTPUT_GROUPED_HUBS}")
    else:
        print(f"  {OUTPUT_H3_HEXAGONS}")
    
    PART3_SUCCESS = False
    final_gdf = None

## Step 3.5: Explore Results (if Part 3 completed)

In [ ]:
if PART3_SUCCESS and final_gdf is not None:
    print("Part 3 Results Summary")
    print("="*80)
    
    # Show columns added
    influence_cols = [col for col in final_gdf.columns if 'pop_' in col or 'emp_' in col or 'terminal' in col]
    print(f"\nColumns added by influence area processing:")
    for col in influence_cols:
        print(f"  - {col}")
    
    # Statistics
    print(f"\nPopulation Statistics:")
    if 'total_pop_influence' in final_gdf.columns:
        print(f"  Total population in influence areas: {final_gdf['total_pop_influence'].sum():,.0f}")
        print(f"  Average per hub: {final_gdf['total_pop_influence'].mean():,.0f}")
        print(f"  Max in single hub: {final_gdf['total_pop_influence'].max():,.0f}")
    
    print(f"\nEmployment Statistics:")
    if 'total_emp_influence' in final_gdf.columns:
        print(f"  Total employment in influence areas: {final_gdf['total_emp_influence'].sum():,.0f}")
        print(f"  Average per hub: {final_gdf['total_emp_influence'].mean():,.0f}")
        print(f"  Max in single hub: {final_gdf['total_emp_influence'].max():,.0f}")
    
    # Bus terminal proximity
    if 'near_bus_terminal' in final_gdf.columns:
        near_terminals = final_gdf['near_bus_terminal'].sum()
        print(f"\nBus Terminal Proximity:")
        print(f"  Hubs within 200m of terminal: {near_terminals}")
        print(f"  Percentage: {near_terminals/len(final_gdf)*100:.1f}%")
    
    # Preview data
    print(f"\nFirst 3 records:")
    display_cols = ['group', 'total_pop_influence', 'total_emp_influence']
    if 'TotalDemand' in final_gdf.columns:
        display_cols.append('TotalDemand')
    if 'near_bus_terminal' in final_gdf.columns:
        display_cols.append('near_bus_terminal')
    
    print(final_gdf[display_cols].head(3))
else:
    print("Part 3 did not complete - no results to display")

## Step 3.6: Export to Excel with Multiple Sheets (Optional)

In [ ]:
if PART3_SUCCESS and final_gdf is not None:
    print("Exporting to Excel with analysis sheets...")
    
    try:
        with pd.ExcelWriter(OUTPUT_FINAL_EXCEL, engine='openpyxl') as writer:
            # Main data sheet
            export_df = final_gdf.copy()
            if 'geometry' in export_df.columns:
                export_df['geometry'] = export_df['geometry'].apply(lambda x: x.wkt if x else None)
            export_df.to_excel(writer, sheet_name='Hub Groups', index=False)
            
            # Top hubs by population
            if 'total_pop_influence' in final_gdf.columns:
                top_pop = final_gdf.nlargest(20, 'total_pop_influence')[[
                    'group', 'total_pop_influence', 'total_emp_influence', 'address'
                ]].copy()
                top_pop.to_excel(writer, sheet_name='Top by Population', index=False)
            
            # Top hubs by employment
            if 'total_emp_influence' in final_gdf.columns:
                top_emp = final_gdf.nlargest(20, 'total_emp_influence')[[
                    'group', 'total_pop_influence', 'total_emp_influence', 'address'
                ]].copy()
                top_emp.to_excel(writer, sheet_name='Top by Employment', index=False)
            
            # Combined score (if demand data available)
            if 'TotalDemand' in final_gdf.columns and 'total_pop_influence' in final_gdf.columns:
                combined = final_gdf.copy()
                # Normalize both metrics to 0-1 scale
                combined['demand_norm'] = (combined['TotalDemand'] - combined['TotalDemand'].min()) / \
                                          (combined['TotalDemand'].max() - combined['TotalDemand'].min())
                combined['pop_norm'] = (combined['total_pop_influence'] - combined['total_pop_influence'].min()) / \
                                       (combined['total_pop_influence'].max() - combined['total_pop_influence'].min())
                combined['combined_score'] = (combined['demand_norm'] + combined['pop_norm']) / 2
                
                top_combined = combined.nlargest(20, 'combined_score')[[
                    'group', 'TotalDemand', 'total_pop_influence', 'combined_score', 'address'
                ]].copy()
                top_combined.to_excel(writer, sheet_name='Top by Combined Score', index=False)
            
            # Zone statistics
            zone_stats = pd.DataFrame({
                'Zone': ['Zone 1 (0-600m)', 'Zone 2 (600-1000m)', 'Zone 3 (1000-1200m)', 'Total'],
                'Avg Population': [
                    final_gdf['pop_zone1'].mean() if 'pop_zone1' in final_gdf.columns else 0,
                    final_gdf['pop_zone2'].mean() if 'pop_zone2' in final_gdf.columns else 0,
                    final_gdf['pop_zone3'].mean() if 'pop_zone3' in final_gdf.columns else 0,
                    final_gdf['total_pop_influence'].mean() if 'total_pop_influence' in final_gdf.columns else 0
                ],
                'Avg Employment': [
                    final_gdf['emp_zone1'].mean() if 'emp_zone1' in final_gdf.columns else 0,
                    final_gdf['emp_zone2'].mean() if 'emp_zone2' in final_gdf.columns else 0,
                    final_gdf['emp_zone3'].mean() if 'emp_zone3' in final_gdf.columns else 0,
                    final_gdf['total_emp_influence'].mean() if 'total_emp_influence' in final_gdf.columns else 0
                ]
            })
            zone_stats.to_excel(writer, sheet_name='Zone Statistics', index=False)
        
        print(f"✓ Excel file created: {OUTPUT_FINAL_EXCEL}")
        print(f"  Sheets: Hub Groups, Top by Population, Top by Employment, Zone Statistics")
        if 'TotalDemand' in final_gdf.columns:
            print(f"          Top by Combined Score")
    
    except Exception as e:
        print(f"⚠ Could not create Excel file: {e}")
        print(f"  CSV file is still available: {OUTPUT_FINAL}")
else:
    print("Skipping Excel export (Part 3 did not complete)")

---
# SUMMARY AND EXPORT
---

In [ ]:
print("="*80)
print("PIPELINE COMPLETE SUMMARY")
print("="*80)

print(f"\nPart 1: H3 Hexagon Processing")
print(f"  ✓ Created {len(gdf_output_part1)} H3 hexagons")
print(f"  ✓ Identified {gdf_output_part1['group'].nunique()} proximity groups")
print(f"  ✓ Output: {OUTPUT_H3_HEXAGONS}")

if PART2_AVAILABLE:
    print(f"\nPart 2: Demand Data Processing")
    print(f"  ✓ Added daily demand data")
    print(f"  ✓ Output: {OUTPUT_GROUPED_HUBS}")
else:
    print(f"\nPart 2: Demand Data Processing")
    print(f"  ✗ Skipped (module not available)")

print(f"\nPart 3: Influence Area Processing")
if PART3_SUCCESS:
    print(f"  ✓ Added population and employment data")
    print(f"  ✓ Created buffer zone statistics")
    if TERMINALS_AVAILABLE:
        print(f"  ✓ Identified terminal proximity")
    print(f"  ✓ Output CSV: {OUTPUT_FINAL}")
    print(f"  ✓ Output Excel: {OUTPUT_FINAL_EXCEL}")
elif PART3_AVAILABLE and not TAZ_AVAILABLE:
    print(f"  ✗ Skipped (TAZ data not available)")
elif not PART3_AVAILABLE:
    print(f"  ✗ Skipped (module not available)")
else:
    print(f"  ✗ Error during processing")

print(f"\n" + "="*80)
print(f"Pipeline complete! Check output files above.")
print("="*80)

# Summary of available outputs
print(f"\nAvailable Output Files:")
print(f"  1. H3 Hexagons: {OUTPUT_H3_HEXAGONS}")
if PART2_AVAILABLE:
    print(f"  2. With Demand: {OUTPUT_GROUPED_HUBS}")
if PART3_SUCCESS:
    print(f"  3. Complete Data (CSV): {OUTPUT_FINAL}")
    print(f"  4. Complete Data (Excel): {OUTPUT_FINAL_EXCEL}")